## Loading and Evaluating a Foundation Model

In [ ]:
# Loading the model
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("gpt2")
print(model)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 769.68it/s, Materializing param=transformer.wte.weight]             


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [ ]:
# Evaluating the model

In [55]:
# Creating a PEFT config
# LoraConfig documentation: https://huggingface.co/docs/peft/main/en/conceptual_guides/lora
from peft import LoraConfig
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"], # typically attention or MLP blocks
    lora_dropout=0.1,
    bias="none", # could be 'none', 'all', or 'lora_only'
    modules_to_save=["lm_head"],
    #use_rslora=True, # try with and without
)

# Evaluating the model

In [67]:
# Loading the dataset
from datasets import load_dataset

splits = ["train", "test"]
ds = {split: ds for split, ds in zip(splits, load_dataset("glue", "rte", split=splits))}
print(ds["train"][0])


{'sentence1': 'No Weapons of Mass Destruction Found in Iraq Yet.', 'sentence2': 'Weapons of Mass Destruction Found in Iraq.', 'label': 1, 'idx': 0}


In [ ]:
# Initializing 

In [56]:
# Combining the two
from peft import get_peft_model
lora_model = get_peft_model(model, config)

In [57]:
# Checking Trainable Parameters of a PEFT Model
lora_model.print_trainable_parameters()

trainable params: 38,892,288 || all params: 163,332,096 || trainable%: 23.8118


In [51]:
# Saving a Trained PEFT Model
lora_model.save_pretrained("gpt-lora")

In [52]:
# Loading a Saved PEFT Model
from peft import AutoPeftModelForCausalLM
lora_model = AutoPeftModelForCausalLM.from_pretrained("gpt-lora")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2711.01it/s, Materializing param=transformer.wte.weight]             


In [53]:
# Generating Text from a PEFT Model
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
inputs = tokenizer("Hello, my name is ", return_tensors="pt")
outputs = model.generate(input_ids=inputs["input_ids"], max_new_tokens=10)
print(tokenizer.batch_decode(outputs))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['Hello, my name is _____. I am a student at the University of']
